In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Dropout, Concatenate,
    GlobalAveragePooling1D, Dense, BatchNormalization,
    LSTM
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import LayerNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, roc_curve, auc,
    accuracy_score, log_loss, matthews_corrcoef, cohen_kappa_score, balanced_accuracy_score
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import SpatialDropout1D

# Load the preprocessed SDN dataset
df = pd.read_csv('Your dataset')

# Ensure the 'label' column exists and is binary
if 'label' not in df.columns:
    raise ValueError("The dataset must contain a 'label' column for binary classification.")

# Select features and labels
features = df.drop(columns=['label'])
labels = df['label']

# Impute missing values
imputer = SimpleImputer(strategy='median')
features_imputed_array = imputer.fit_transform(features)

# Check the shape of features before and after imputation
print(f"Shape before imputation: {features.shape}")
print(f"Shape after imputation: {features_imputed_array.shape}")

# Reconstruct the cleaned dataframe
if features_imputed_array.shape[1] != features.shape[1]:
    print("Warning: Column count mismatch. Adjusting column names accordingly.")
    features_imputed = pd.DataFrame(features_imputed_array, columns=[f'feature_{i}' for i in range(features_imputed_array.shape[1])])
else:
    features_imputed = pd.DataFrame(features_imputed_array, columns=features.columns)

# Combine the imputed features with the labels
df_cleaned = pd.concat([features_imputed, labels], axis=1)

# Split features and labels again after imputation
X = df_cleaned.drop(columns=['label']).values
y = df_cleaned['label'].values

# Determine the number of features
input_length = X.shape[1]
print(f"Number of features: {input_length}")

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train.reshape((X_train.shape[0], input_length, 1))
X_test = X_test.reshape((X_test.shape[0], input_length, 1))



# Define the CNN-LSTM Model with increased complexity
input_layer = Input(shape=(input_length, 1))

# Initial Conv1D Layers with increased filters and regularization
x = Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', kernel_regularizer=l2(0.001))(input_layer)
x = Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', kernel_regularizer=l2(0.001))(x)
x = MaxPooling1D(pool_size=2)(x)
x = Dropout(0.5)(x)

# Inception Block
branch1 = Conv1D(filters=32, kernel_size=1, activation='relu', padding='same', kernel_regularizer=l2(0.001))(x)
branch2 = Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', kernel_regularizer=l2(0.001))(x)
branch3 = Conv1D(filters=32, kernel_size=5, activation='relu', padding='same', kernel_regularizer=l2(0.001))(x)
x = Concatenate()([branch1, branch2, branch3])
x = MaxPooling1D(pool_size=2)(x)
x = Dropout(0.3)(x)

# Add Spatial Dropout before LSTM for regularization across feature dimensions
x = SpatialDropout1D(0.2)(x)

# LSTM layer with increased units and regularization
x = LSTM(256, return_sequences=False, kernel_regularizer=l2(0.001))(x)

# Flatten and Dense Layers
x = Dense(units=256, activation='relu', kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)
x = Dense(units=256, activation='relu', kernel_regularizer=l2(0.001))(x)
x = BatchNormalization()(x)

# Output Layer for Binary Classification
output_layer = Dense(units=1, activation='sigmoid')(x)

# Define the Model
model = Model(inputs=input_layer, outputs=output_layer)

# Compile the Model
optimizer = Adam(learning_rate=5e-4)  # Reduce learning rate for finer updates
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Adjust Early Stopping patience
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

callbacks = [early_stopping, scheduler, model_checkpoint]

# Train the Model with increased epochs
history = model.fit(
    X_train,
    y_train,
    epochs=200,  # Increased epochs
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=callbacks
)


# Evaluate the Model
# Measure Inference Time
import time

start_time = time.time()
y_pred_proba = model.predict(X_test)
inference_time = time.time() - start_time
print(f"Inference time: {inference_time:.4f} seconds")

# Convert predicted probabilities to binary class labels
y_pred = (y_pred_proba > 0.5).astype("int32").flatten()
y_true = y_test.flatten()

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

# Classification Report
report = classification_report(y_true, y_pred, zero_division=1, output_dict=True)
print("Classification Report:\n", report)

# Extract Metrics
precision = report['weighted avg']['precision']
f1_score = report['weighted avg']['f1-score']
tpr = report['weighted avg']['recall']  # Recall is the same as TPR
print(f"TPR: {tpr:.4f}, Precision: {precision:.4f}, F1-Score: {f1_score:.4f}")

# ROC-AUC and ROC Curve
roc_auc = roc_auc_score(y_true, y_pred_proba)
print(f"ROC-AUC: {roc_auc:.4f}")

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
roc_auc_value = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc_value:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

# Additional Metrics
accuracy = accuracy_score(y_true, y_pred)
logloss = log_loss(y_true, y_pred_proba)
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)
balanced_acc = balanced_accuracy_score(y_true, y_pred)

# Specificity
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
specificity = tn / (tn + fp)

print(f"Accuracy: {accuracy:.4f}")
print(f"Log Loss: {logloss:.4f}")
print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")
print(f"Specificity (TNR): {specificity:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Number of Parameters: {model.count_params()}")
